In [0]:
%run ./00_setup_config

In [0]:
from pyspark.sql import functions as F


def _ler(schema=None):
    leitor = spark.read.options(**adls_options)
    if schema:
        leitor = leitor.schema(schema)
    df = leitor.parquet(CAMINHO_ORIGEM)
    df.schema  # forca a analise: erros de acesso saem aqui, nao numa acao posterior
    return df


def ler_origem():
    try:
        try:
            return _ler()
        except Exception as e:
            if "NANOS" not in str(e):
                raise
            print("[info] datas em TIMESTAMP(NANOS): relendo com SCHEMA_DT_INTEIRO")
            return _ler(SCHEMA_DT_INTEIRO)
    except Exception as e:
        msg = str(e)
        if "PATH_NOT_FOUND" in msg or "Path does not exist" in msg or "does not exist" in msg:
            raise FileNotFoundError(
                f"Nenhum arquivo {TABELA}.parquet em {CONTAINER}/{PREFIXO}/.\n"
                "A janela de tempo real pode ainda nao ter sido aberta."
            ) from None
        if "AuthorizationPermissionMismatch" in msg or "403" in msg:
            raise PermissionError(
                "403: o service principal autenticou, mas nao tem leitura no container."
            ) from None
        if "AADSTS" in msg or "invalid_client" in msg or "401" in msg:
            raise PermissionError(
                "401: credencial recusada - conferir ADLS_CLIENT_ID/SECRET/TENANT_ID no .env."
            ) from None
        raise


df_bruto = ler_origem()

# Datas lidas como inteiro (nanossegundos) viram timestamp_ntz: horario de
# parede, sem fuso - o mesmo significado do TIMESTAMP(NANOS, isAdjustedToUTC=false)
# da origem. Soma de intervalo sobre TIMESTAMP_NTZ nao depende do fuso da
# sessao. Dias e segundos separados porque o total de microssegundos nao
# cabe num int - timestampadd(MICROSECOND, ...) estoura e volta a 1969.
NS_DIA = 86_400_000_000_000
for coluna, tipo in df_bruto.dtypes:
    if coluna.startswith(PREFIXO_DATA) and tipo == "bigint":
        df_bruto = df_bruto.withColumn(coluna, F.expr(
            f"TIMESTAMP_NTZ'1970-01-01 00:00:00' + make_dt_interval("
            f"CAST({coluna} div {NS_DIA} AS INT), 0, 0, "
            f"CAST(({coluna} % {NS_DIA}) / 1000000000 AS DECIMAL(18, 6)))"
        ))

print(f"[ok] leitura distribuida: {len(df_bruto.columns)} colunas")
df_bruto.printSchema()

In [0]:
df_pedidos = df_bruto.withColumn(
    "dt_lote",
    F.to_timestamp_ntz(
        F.regexp_extract(F.col("_metadata.file_path"), r"(\d{4}/\d{2}/\d{2}/\d{6})/", 1),
        F.lit("yyyy/MM/dd/HHmmss"),
    ),
)

resumo = df_pedidos.agg(
    F.count("*").alias("linhas"),
    F.countDistinct("dt_lote").alias("lotes"),
    F.min("dt_lote").alias("primeiro_lote"),
    F.max("dt_lote").alias("ultimo_lote"),
    F.sum(F.col("dt_lote").isNull().cast("int")).alias("sem_lote"),
).first()

print(f"Linhas        : {resumo['linhas']}")
print(f"Lotes lidos   : {resumo['lotes']}")
print(f"Primeiro lote : {resumo['primeiro_lote']}")
print(f"Ultimo lote   : {resumo['ultimo_lote']}")
if resumo["sem_lote"]:
    print(f"[!] {resumo['sem_lote']} linhas sem carimbo de lote - caminho fora do padrao")

In [0]:
display(df_pedidos.limit(10))